# Memory Task Design Checks

Pair structure, probe assignment, slider counterbalancing, pair-order randomisation, and image-to-position mapping.

In [1]:
import re, json, hashlib, os, math, warnings, textwrap, sys
from pathlib import Path
from datetime import datetime
from collections import Counter, OrderedDict
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# Import shared JS parsers
sys.path.insert(0, str(Path(".").resolve()))
from js_import import (load_sources, parse, save_fig, save_html, apply_style,
                        REPO_ROOT, DESIGN_DIR, FIG_DIR, sha256)
apply_style()

# ── Load all JS sources ──
stimuli_src = (REPO_ROOT / "static/task/stimuli.js").read_text() if (REPO_ROOT / "static/task/stimuli.js").exists() else ""
memory_src  = (REPO_ROOT / "static/task/memory_task.js").read_text() if (REPO_ROOT / "static/task/memory_task.js").exists() else ""
details_src = (REPO_ROOT / "static/task/stimuli-details.js").read_text() if (REPO_ROOT / "static/task/stimuli-details.js").exists() else ""
trial_src   = (REPO_ROOT / "static/task/trial.js").read_text() if (REPO_ROOT / "static/task/trial.js").exists() else ""
index_src   = (REPO_ROOT / "index.html").read_text() if (REPO_ROOT / "index.html").exists() else ""

# Re-export original helper functions from the shared module
parse_js_flat_array = parse.flat_array
parse_js_2d_array = parse.array_2d
parse_js_string = parse.string
parse_js_string_array = parse.string_array
parse_js_int_pair_array = parse.int_pair_array

# ── Original helper functions that stay local ──
def read_text(relpath):
    p = REPO_ROOT / relpath
    return p.read_text(encoding="utf-8") if p.exists() else None

def pair_in(pair, plist):
    return any(pair[0] == p[0] and pair[1] == p[1] for p in plist)

def styled_table_css(uid="tbl"):
    return textwrap.dedent(f"""\
    <style>
    #{uid} th {{ background:#f3f3f3; color:#222; font-weight:650;
      border:1px solid #d6d6d6; padding:5px 10px; text-align:left; }}
    #{uid} td {{ border:1px solid #e1e1e1; padding:4px 10px;
      font-variant-numeric:tabular-nums; }}
    #{uid} tr:nth-child(even) td {{ background:#fafafa; }}
    #{uid} table {{ border-collapse:collapse; font-size:12px;
      font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Arial,sans-serif; }}
    #{uid} caption {{ caption-side:top; padding:4px 0; color:#444;
      font-size:12px; text-align:left; font-weight:600; }}
    </style>
    """)

# save_html is imported from js_import (category-aware)

print("Setup complete.")


Setup complete.


In [2]:
# ── Variables shared with 01_game_trials ──
factors_vol = parse_js_flat_array(stimuli_src, "factors_vol")
factors_stc = parse_js_flat_array(stimuli_src, "factors_stc")
fv_m = re.search(r"var\s+factors_valence\s*=\s*\[(.*?)\]", stimuli_src)
factors_valence = re.findall(r"'(\w+)'", fv_m.group(1)) if fv_m else []

# Change-point arrays (trial indices where drone jumps substantially)
CP_HIGH_VOL = parse_js_flat_array(stimuli_src, "CP_HIGH_VOL")
CP_LOW_VOL  = parse_js_flat_array(stimuli_src, "CP_LOW_VOL")
if not CP_HIGH_VOL:
    # Compute from trajectory if not explicitly defined
    main_drone = parse_js_2d_array(stimuli_src, "main_drone_position")
    if main_drone:
        def _compute_cps(blk):
            d = np.array(main_drone[blk])
            jumps = np.abs(np.diff(d))
            thr = np.percentile(jumps, 75)
            return [t+2 for t, j in enumerate(jumps) if j > thr]
        hv_blk = next((i for i in range(4) if factors_vol[i] == 49), 0)
        lv_blk = next((i for i in range(4) if factors_vol[i] == 4), 1)
        CP_HIGH_VOL = _compute_cps(hv_blk)
        CP_LOW_VOL  = _compute_cps(lv_blk)

EXAMPLE_BLOCK = 1  # used in pair-coverage visualisation
print(f"CP_HIGH_VOL: {len(CP_HIGH_VOL) if CP_HIGH_VOL else 0} change-points")
print(f"CP_LOW_VOL:  {len(CP_LOW_VOL) if CP_LOW_VOL else 0} change-points")

# Stimulus image pool
STIMULUS_IMAGE_POOL = parse_js_string_array(stimuli_src, "STIMULUS_IMAGE_POOL")
if not STIMULUS_IMAGE_POOL:
    # Fallback: glob from static/stimuli/
    stim_dir = REPO_ROOT / "static" / "stimuli"
    if stim_dir.exists():
        STIMULUS_IMAGE_POOL = sorted([str(p.relative_to(REPO_ROOT)) for p in stim_dir.glob("*.png")])
print(f"STIMULUS_IMAGE_POOL: {len(STIMULUS_IMAGE_POOL) if STIMULUS_IMAGE_POOL else 0} items")

stim_version = parse_js_string(stimuli_src, "stimuli_version") or "unknown"
print(f"stimuli_version: {stim_version}")


CP_HIGH_VOL: 12 change-points
CP_LOW_VOL:  12 change-points
STIMULUS_IMAGE_POOL: 200 items
stimuli_version: v10-png-main-practice-emoji


## F. Memory-pair structure

In [3]:
PREDEFINED_PAIRS = parse_js_int_pair_array(memory_src, "PREDEFINED_PAIRS")
SLIDER_PAIRS     = parse_js_int_pair_array(stimuli_src, "SLIDER_PAIRS")
BOUNDARY_MIDDLE  = parse_js_int_pair_array(stimuli_src, "BOUNDARY_MIDDLE_PAIRS")
NONBOUNDARY_MID  = parse_js_int_pair_array(stimuli_src, "NONBOUNDARY_MIDDLE_PAIRS")

print(f"PREDEFINED_PAIRS ({len(PREDEFINED_PAIRS)}): {PREDEFINED_PAIRS}")

def probe_candidate_indices(pair):
    """Return all serial positions between the two endpoint trials."""
    return list(range(pair[0] + 1, pair[1]))

def probe_distance_label(n_candidates):
    if n_candidates == 1:
        return "1 interveneing object"
    if n_candidates == 2:
        return "2 intervening objects"
    return f"d{n_candidates} intervening objects"

def placement_design_label(pair):
    candidates = probe_candidate_indices(pair)
    n = len(candidates)

    if n == 1:
        return "middle probe"
    if n == 2:
        return "earlier or later probe (counterbalanced)"
    return "earlier or later probe (counterbalanced)"

pair_rows = []

for pid, pair in enumerate(PREDEFINED_PAIRS):
    candidates = probe_candidate_indices(pair)
    n_candidates = len(candidates)

    is_slider = pair_in(pair, SLIDER_PAIRS)
    is_bnd    = pair_in(pair, BOUNDARY_MIDDLE)
    is_nbnd   = pair_in(pair, NONBOUNDARY_MID)

    pair_rows.append(dict(
        pair_id=f"P{pid+1:02d}",
        pair=f"({pair[0]}, {pair[1]})",
        endpoint_1=pair[0],
        endpoint_2=pair[1],
        true_distance=n_candidates,
        candidate_indices=str(candidates),
        n_candidates=n_candidates,

        # New design labels
        design_role=placement_design_label(pair),
        probe_distance_type=probe_distance_label(n_candidates),

        # Keep these booleans for possible downstream filtering,
        # but do not display them in the main reviewer table.
        legacy_slider_pair=is_slider,
        legacy_boundary_middle=is_bnd,
        legacy_nonboundary_middle=is_nbnd,
    ))

pair_df = pd.DataFrame(pair_rows)

display_cols = [
    "pair_id",
    "pair",
    "true_distance",
    "n_candidates",
    "candidate_indices",
    "design_role",
    "probe_distance_type",
]

display(pair_df[display_cols])

assert len(PREDEFINED_PAIRS) == 14
d1 = pair_df[pair_df.n_candidates == 1]
d2 = pair_df[pair_df.n_candidates == 2]
assert len(d1) == 6 and len(d2) == 8
assert (pair_df["design_role"].isin([
    "middle probe",
    "earlier or later probe (counterbalanced)",
    "earlier or later probe (counterbalanced)",
])).all()

PREDEFINED_PAIRS (14): [[2, 4], [6, 9], [7, 10], [11, 13], [16, 18], [17, 20], [22, 24], [23, 26], [28, 31], [30, 33], [34, 36], [35, 38], [39, 41], [42, 45]]


,pair_id,pair,true_distance,n_candidates,candidate_indices,design_role,probe_distance_type
0,P01,"(2, 4)",1,1,[3],middle probe,1 interveneing object
1,P02,"(6, 9)",2,2,"[7, 8]",earlier or later probe (counterbalanced),2 intervening objects
2,P03,"(7, 10)",2,2,"[8, 9]",earlier or later probe (counterbalanced),2 intervening objects
3,P04,"(11, 13)",1,1,[12],middle probe,1 interveneing object
4,P05,"(16, 18)",1,1,[17],middle probe,1 interveneing object
5,P06,"(17, 20)",2,2,"[18, 19]",earlier or later probe (counterbalanced),2 intervening objects
6,P07,"(22, 24)",1,1,[23],middle probe,1 interveneing object
7,P08,"(23, 26)",2,2,"[24, 25]",earlier or later probe (counterbalanced),2 intervening objects
8,P09,"(28, 31)",2,2,"[29, 30]",earlier or later probe (counterbalanced),2 intervening objects
9,P10,"(30, 33)",2,2,"[31, 32]",earlier or later probe (counterbalanced),2 intervening objects


### Memory-pair coverage along the trial axis
Horizontal bars show each pair's span; **\u00d7** marks the slider probe item(s).
Vertical lines indicate high-vol change points.

In [4]:
# -----------------------------
# Configurable setting
# -----------------------------
EXAMPLE_SEED = 20260513
EXAMPLE_BLOCK = 1
rng = np.random.default_rng(EXAMPLE_SEED)

# Use high-vol CP for block 1/3 and low-vol CP for block 2/4.
# If CP_HIGH_VOL / CP_LOW_VOL are already defined earlier, this reuses them.
block_is_high_vol = EXAMPLE_BLOCK in [1, 3]
cp_list = CP_HIGH_VOL if block_is_high_vol else CP_LOW_VOL
cp_label = "high-vol change point" if block_is_high_vol else "low-vol change point"

# -----------------------------
# Match memory_task.js logic
# -----------------------------
def pair_key(pair):
    return f"{pair[0]}_{pair[1]}"

def get_intervening_indices(pair):
    return list(range(pair[0] + 1, pair[1]))

def get_distance2_pairs(pairs):
    return [p for p in pairs if len(get_intervening_indices(p)) == 2]

def init_placement_probe_assignments_for_block(pairs, rng):
    """
    Python mirror of memory_task.js:
    - Take all distance-2 pairs.
    - Shuffle them.
    - Alternate earlier/later probe assignment.
    """
    assignments = {}
    distance2_pairs = get_distance2_pairs(pairs)
    shuffled_idx = rng.permutation(len(distance2_pairs))
    shuffled_pairs = [distance2_pairs[i] for i in shuffled_idx]

    for i, pair in enumerate(shuffled_pairs):
        candidates = get_intervening_indices(pair)
        choose_earlier = (i % 2 == 0)
        assignments[pair_key(pair)] = candidates[0] if choose_earlier else candidates[1]

    return assignments

def choose_placement_probe_index(pair, assignments):
    candidates = get_intervening_indices(pair)

    if len(candidates) == 0:
        return None

    # One-intervening-item pairs are deterministic.
    if len(candidates) == 1:
        return candidates[0]

    # Two-intervening-item pairs use the balanced block-level assignment.
    if len(candidates) == 2:
        return assignments[pair_key(pair)]

    # Fallback only; current design never reaches this.
    return candidates[0]

# Pair order is also pseudo-randomized within block.
pair_order_idx = rng.permutation(len(PREDEFINED_PAIRS))
ordered_pairs = [PREDEFINED_PAIRS[i] for i in pair_order_idx]

# Placement-probe assignment is pseudo-randomized separately,
# then balanced by alternating earlier/later assignment among distance-2 pairs.
probe_assignments = init_placement_probe_assignments_for_block(PREDEFINED_PAIRS, rng)

example_rows = []

for mem_pos, pair in enumerate(ordered_pairs, start=1):
    pair_id_num = PREDEFINED_PAIRS.index(pair) + 1
    candidates = get_intervening_indices(pair)
    selected_probe = choose_placement_probe_index(pair, probe_assignments)

    if len(candidates) == 1:
        probe_position_label = "center"
    elif selected_probe == candidates[0]:
        probe_position_label = "earlier"
    elif selected_probe == candidates[1]:
        probe_position_label = "later"
    else:
        probe_position_label = "fallback"

    true_position_pct = (
        None if selected_probe is None
        else 100 * (selected_probe - pair[0]) / (pair[1] - pair[0])
    )

    example_rows.append(dict(
        memory_order_position=mem_pos,
        pair_id=f"P{pair_id_num:02d}",
        pair=f"({pair[0]}, {pair[1]})",
        endpoint_1=pair[0],
        endpoint_2=pair[1],
        candidate_indices=str(candidates),
        selected_probe_index=selected_probe,
        selected_probe_position=probe_position_label,
        true_position_pct=true_position_pct,
        design_role=placement_design_label(pair),
    ))

example_probe_df = pd.DataFrame(example_rows)

example_display_cols = [
    "memory_order_position",
    "pair_id",
    "pair",
    "candidate_indices",
    "selected_probe_index",
    "selected_probe_position",
    "true_position_pct",
    "design_role",
]

display(example_probe_df[example_display_cols])

# -----------------------------
# Balance check for this example block
# -----------------------------
balance_summary = (
    example_probe_df
    .groupby("selected_probe_position")
    .size()
    .reindex(["center", "earlier", "later"], fill_value=0)
    .rename("count")
    .reset_index()
)

display(balance_summary)

assert balance_summary.loc[
    balance_summary.selected_probe_position == "center", "count"
].iloc[0] == 6

assert balance_summary.loc[
    balance_summary.selected_probe_position == "earlier", "count"
].iloc[0] == 4

assert balance_summary.loc[
    balance_summary.selected_probe_position == "later", "count"
].iloc[0] == 4

print("✓ Example block has 6 center probes, 4 earlier probes, and 4 later probes.")
print("✓ Pair order is pseudo-randomized.")
print("✓ Distance-2 probe assignment is pseudo-randomized and balanced.")

# -----------------------------
# Publication-level visualization
# -----------------------------
fig, ax = plt.subplots(figsize=(16, 7.2))

# Change-point background lines
for cp in cp_list:
    ax.axvline(
        cp,
        color="#c9a227",
        ls="-",
        lw=1.3,
        alpha=0.35,
        zorder=1
    )

# Color by selected probe role.
role_colors = {
    "center": "#2f4b7c",   # single intervening candidate
    "earlier": "#a05195",  # earlier of two candidates
    "later": "#f95d6a",    # later of two candidates
}

role_labels = {
    "center": "single intervening item",
    "earlier": "earlier probe of the intervening two",
    "later": "later probe of the intervening two",
}

used_labels = set()

# Plot in memory-test order: y-axis is randomized order position.
for _, row in example_probe_df.iterrows():
    y = len(example_probe_df) - row.memory_order_position + 1
    ep1 = int(row.endpoint_1)
    ep2 = int(row.endpoint_2)
    probe = int(row.selected_probe_index)
    role = row.selected_probe_position

    color = role_colors.get(role, "#666666")
    label = role_labels.get(role, role)
    plot_label = label if label not in used_labels else None
    used_labels.add(label)

    # Span between endpoints
    ax.plot(
        [ep1, ep2],
        [y, y],
        color=color,
        lw=4.0,
        alpha=0.82,
        solid_capstyle="round",
        label=plot_label,
        zorder=3
    )

    # Endpoint items: circles
    ax.scatter(
        [ep1, ep2],
        [y, y],
        color=color,
        s=70,
        edgecolor="white",
        linewidth=0.8,
        zorder=4
    )

    # Candidate intervening items: faint vertical ticks
    candidates = get_intervening_indices([ep1, ep2])
    for c in candidates:
        ax.scatter(
            c,
            y,
            marker="|",
            color=color,
            s=230,
            lw=2.0,
            alpha=0.35,
            zorder=4
        )

    # Selected probe item: bold X
    ax.scatter(
        probe,
        y,
        marker="x",
        color="black",
        s=130,
        lw=2.8,
        zorder=6
    )

    # Left-side row label
    ax.text(
        -0.25,
        y,
        f"{row.memory_order_position:02d}  {row.pair_id}  {row.pair}",
        ha="right",
        va="center",
        fontsize=9,
        fontfamily="monospace"
    )

    # Right-side probe label
    ax.text(
        51.2,
        y,
        f"probe {probe} · {role} · {row.true_position_pct:.1f}%",
        ha="left",
        va="center",
        fontsize=8.5,
        color="#333333"
    )

# Custom legend elements
legend_handles = [
    Line2D([0], [0], color="#2f4b7c", lw=4, label="single intervening item"),
    Line2D([0], [0], color="#a05195", lw=4, label="earlier probe of two"),
    Line2D([0], [0], color="#f95d6a", lw=4, label="later probe of two"),
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor="#555555",
        markeredgecolor="white",
        markersize=8,
        label="endpoint item"
    ),
    Line2D(
        [0], [0],
        marker="|",
        color="#555555",
        linestyle="None",
        markersize=14,
        alpha=0.45,
        label="candidate intervening item"
    ),
    Line2D(
        [0], [0],
        marker="x",
        color="black",
        linestyle="None",
        markersize=9,
        markeredgewidth=2.5,
        label="selected slider probe"
    ),
    Line2D(
        [0], [0],
        color="#c9a227",
        lw=2,
        alpha=0.55,
        label=cp_label
    ),
]

ax.legend(
    handles=legend_handles,
    fontsize=9,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.10),
    frameon=False,
    ncol=4,
    handlelength=2.2,
    columnspacing=1.2
)

ax.set_xlim(0, 56)
ax.set_ylim(0.2, len(example_probe_df) + 0.8)
ax.set_xlabel("Trial serial position within block", fontsize=11)
ax.set_yticks([])
ax.set_ylabel("")

ax.set_title(
    "Memory-pair coverage and selected slider probes",
    fontsize=14,
    fontweight="bold",
    pad=14
)

ax.text(
    0,
    len(example_probe_df) + 0.45,
    f"Example participant/block visualization · block {EXAMPLE_BLOCK} · seed {EXAMPLE_SEED}. "
    "Rows are randomized memory-test order; × marks the item actually placed on the slider.",
    fontsize=10.5,
    color="#444444"
)

# Light grid only on x-axis
ax.set_xticks(range(0, 51, 5))
ax.grid(axis="x", color="#dddddd", linewidth=0.8, alpha=0.55)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_alpha(0.35)

fig.tight_layout()

save_fig(fig, "memory_pair_coverage", category="memory", also_html=True)

# Explicit HTML export columns so hidden/internal columns do not leak into reviewer-facing output.
save_html(
    "<h2>Memory Pair Candidate and Probe Structure</h2>"
    "<p>Every predefined memory pair receives a slider probe. "
    "For one-intervening-item pairs, the only intervening item is probed. "
    "For two-intervening-item pairs, the earlier/later probe assignment is pseudo-randomized "
    "and balanced within block.</p>"
    "<h3>Example participant/block probe assignment</h3>"
    + example_probe_df[example_display_cols].to_html(index=False)
    + "<h3>Probe balance for this example block</h3>"
    + balance_summary.to_html(index=False),
    "memory_pair_candidate_structure.html"
, category="memory")

plt.show()

,memory_order_position,pair_id,pair,candidate_indices,selected_probe_index,selected_probe_position,true_position_pct,design_role
0,1,P06,"(17, 20)","[18, 19]",19,later,66.666667,earlier or later probe (counterbalanced)
1,2,P08,"(23, 26)","[24, 25]",25,later,66.666667,earlier or later probe (counterbalanced)
2,3,P03,"(7, 10)","[8, 9]",9,later,66.666667,earlier or later probe (counterbalanced)
3,4,P11,"(34, 36)",[35],35,center,50.000000,middle probe
4,5,P10,"(30, 33)","[31, 32]",31,earlier,33.333333,earlier or later probe (counterbalanced)
5,6,P01,"(2, 4)",[3],3,center,50.000000,middle probe
6,7,P04,"(11, 13)",[12],12,center,50.000000,middle probe
7,8,P05,"(16, 18)",[17],17,center,50.000000,middle probe
8,9,P02,"(6, 9)","[7, 8]",8,later,66.666667,earlier or later probe (counterbalanced)
9,10,P09,"(28, 31)","[29, 30]",29,earlier,33.333333,earlier or later probe (counterbalanced)


,selected_probe_position,count
0,center,6
1,earlier,4
2,later,4


✓ Example block has 6 center probes, 4 earlier probes, and 4 later probes.
✓ Pair order is pseudo-randomized.
✓ Distance-2 probe assignment is pseudo-randomized and balanced.
  ✓ memory_task/memory_pair_coverage.png + .pdf
  ✓ memory_task/memory_pair_coverage.html
  ✓ memory_task/memory_pair_candidate_structure.html


## G. Slider-probe counterbalancing simulation

Every pair now gets a slider probe. Distance-1 pairs use the only candidate.
Distance-2 pairs balance earlier/later by shuffling the 8 distance-2 pairs
and alternating.

In [5]:
# ============================================================
# Slider-probe counterbalancing simulation
# ============================================================
# Purpose:
#   Validate the stochastic part of the revised memory-test design.
#
#   For every simulated participant and every block:
#   - all 14 predefined memory pairs receive a slider probe
#   - 6 distance-1 pairs use the only intervening item ("center")
#   - 8 distance-2 pairs are split into 4 earlier probes and 4 later probes
#
#   Across simulated participants:
#   - earlier/later assignment for each distance-2 pair should be approximately balanced
#   - pair order should vary, but this section focuses on probe-balance, not pair-order distribution

NOTEBOOK_SEED = 42
N_SIM = 1000   # Use 50 for quick debugging; 1000+ is better for reviewer-facing stability.

rng_master = np.random.default_rng(NOTEBOOK_SEED)

# -----------------------------
# Helpers mirroring memory_task.js
# -----------------------------
def get_intervening(pair):
    """Return all serial positions between the two endpoint trials."""
    return list(range(pair[0] + 1, pair[1]))

def get_d2_pairs():
    """Distance-2 pairs have exactly two intervening candidate probes."""
    return [p for p in PREDEFINED_PAIRS if len(get_intervening(p)) == 2]

def pair_to_label(pair):
    """Stable pair label for tables/plots."""
    return f"P{PREDEFINED_PAIRS.index(pair) + 1:02d} {tuple(pair)}"

def simulate_block(block, rng):
    """
    Python mirror of memory_task.js placement logic.

    Runtime logic being validated:
    1. Memory-pair order is shuffled within block.
    2. Distance-2 pairs are separately shuffled.
    3. The shuffled distance-2 list is alternated earlier/later.
    4. Distance-1 pairs always use their only intervening item.
    """
    # Pair order pseudo-randomization
    order = PREDEFINED_PAIRS.copy()
    rng.shuffle(order)

    # Distance-2 probe assignment pseudo-randomization
    d2_pairs = get_d2_pairs()
    d2_shuffled = d2_pairs.copy()
    rng.shuffle(d2_shuffled)

    d2_assignment = {}
    for i, pair in enumerate(d2_shuffled):
        candidates = get_intervening(pair)
        choose_earlier = (i % 2 == 0)
        d2_assignment[tuple(pair)] = candidates[0] if choose_earlier else candidates[1]

    rows = []

    for mem_pos, pair in enumerate(order, start=1):
        candidates = get_intervening(pair)

        if len(candidates) == 1:
            probe_idx = candidates[0]
            probe_label = "center"
        elif len(candidates) == 2:
            probe_idx = d2_assignment[tuple(pair)]
            probe_label = "earlier" if probe_idx == candidates[0] else "later"
        else:
            # Current design should never reach this branch.
            probe_idx = candidates[0] if candidates else None
            probe_label = "fallback"

        true_pos_pct = (
            None if probe_idx is None
            else 100 * (probe_idx - pair[0]) / (pair[1] - pair[0])
        )

        rows.append(dict(
            block=block,
            memory_order_position=mem_pos,
            pair_id=f"P{PREDEFINED_PAIRS.index(pair) + 1:02d}",
            pair=tuple(pair),
            pair_label=pair_to_label(pair),
            endpoint_1=pair[0],
            endpoint_2=pair[1],
            candidate_indices=tuple(candidates),
            n_candidates=len(candidates),
            selected_probe_index=probe_idx,
            selected_probe_position=probe_label,
            selected_probe_ordinal=(candidates.index(probe_idx) + 1) if probe_idx in candidates else None,
            true_position_pct=round(true_pos_pct, 2) if true_pos_pct is not None else None,
        ))

    return rows

# -----------------------------
# Run simulation
# -----------------------------
all_rows = []

for sim_participant in range(N_SIM):
    # Separate participant-level RNG so each simulated participant is reproducible.
    rng = np.random.default_rng(NOTEBOOK_SEED + sim_participant)

    for block in range(1, 5):
        block_rows = simulate_block(block, rng)
        for row in block_rows:
            row["sim_participant"] = sim_participant
            all_rows.append(row)

sim_df = pd.DataFrame(all_rows)

# -----------------------------
# Hard invariant checks
# -----------------------------
for sim_participant in range(N_SIM):
    for block in range(1, 5):
        bdf = sim_df[
            (sim_df.sim_participant == sim_participant) &
            (sim_df.block == block)
        ]

        assert len(bdf) == 14
        assert (bdf.selected_probe_position == "center").sum() == 6
        assert (bdf.selected_probe_position == "earlier").sum() == 4
        assert (bdf.selected_probe_position == "later").sum() == 4
        assert bdf.pair_id.nunique() == 14

print(f"✓ {N_SIM} simulated participants × 4 blocks checked.")
print("✓ Every participant-block has 14 slider trials.")
print("✓ Every participant-block has exact probe balance: 6 center / 4 earlier / 4 later.")
print("✓ Every pair appears once per participant-block.")

# -----------------------------
# Summary tables
# -----------------------------
participant_block_summary = (
    sim_df
    .groupby(["sim_participant", "block", "selected_probe_position"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["center", "earlier", "later"]:
    if col not in participant_block_summary.columns:
        participant_block_summary[col] = 0

summary_by_block = (
    participant_block_summary
    .groupby("block")[["center", "earlier", "later"]]
    .agg(["mean", "min", "max"])
)

display(summary_by_block)

# Distance-2 pair-level assignment rate across simulations.
d2_rate_df = (
    sim_df[sim_df.n_candidates == 2]
    .groupby(["block", "pair_id", "pair_label", "selected_probe_position"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["earlier", "later"]:
    if col not in d2_rate_df.columns:
        d2_rate_df[col] = 0

d2_rate_df["total"] = d2_rate_df["earlier"] + d2_rate_df["later"]
d2_rate_df["p_earlier"] = d2_rate_df["earlier"] / d2_rate_df["total"]
d2_rate_df["p_later"] = d2_rate_df["later"] / d2_rate_df["total"]
d2_rate_df["earlier_minus_later"] = d2_rate_df["earlier"] - d2_rate_df["later"]

display(d2_rate_df[[
    "block",
    "pair_id",
    "pair_label",
    "earlier",
    "later",
    "total",
    "p_earlier",
    "p_later",
    "earlier_minus_later",
]])

# ============================================================
# Plot 1: exact within-block balance
# ============================================================
balance_long = (
    participant_block_summary
    .melt(
        id_vars=["sim_participant", "block"],
        value_vars=["center", "earlier", "later"],
        var_name="probe_role",
        value_name="count"
    )
)

plot_summary = (
    balance_long
    .groupby(["block", "probe_role"])["count"]
    .agg(["mean", "min", "max"])
    .reset_index()
)

role_order = ["center", "earlier", "later"]
role_colors = {
    "center": "#2f4b7c",
    "earlier": "#a05195",
    "later": "#f95d6a",
}
role_labels = {
    "center": "single intervening item",
    "earlier": "earlier of two intervening items",
    "later": "later of two intervening items",
}

fig, ax = plt.subplots(figsize=(10.5, 5.8))

x = np.arange(1, 5)
bottom = np.zeros(len(x))

for role in role_order:
    vals = (
        plot_summary[plot_summary.probe_role == role]
        .sort_values("block")["mean"]
        .values
    )

    ax.bar(
        x,
        vals,
        bottom=bottom,
        color=role_colors[role],
        edgecolor="white",
        linewidth=1.0,
        label=role_labels[role],
        width=0.68,
    )

    # Segment labels
    for xi, btm, val in zip(x, bottom, vals):
        ax.text(
            xi,
            btm + val / 2,
            f"{int(val)}",
            ha="center",
            va="center",
            color="white",
            fontsize=11,
            fontweight="bold"
        )

    bottom += vals

# Total labels
for xi, total in zip(x, bottom):
    ax.text(
        xi,
        total + 0.35,
        f"{int(total)} total",
        ha="center",
        va="bottom",
        fontsize=10,
        color="#333333"
    )

ax.set_xticks(x)
ax.set_xticklabels([f"Block {i}" for i in x])
ax.set_ylim(0, 16)
ax.set_ylabel("Slider-probe trials per block", fontsize=11)
ax.set_title(
    "Exact within-block slider-probe balance",
    fontsize=14,
    fontweight="bold",
    pad=14
)

ax.text(
    0.02,
    0.96,
    f"N = {N_SIM} simulated participants. Each participant-block is constrained to 6 center, 4 earlier, and 4 later probes.",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    color="#444444"
)

ax.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=3,
    fontsize=9
)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#dddddd", linewidth=0.8, alpha=0.45)
ax.set_axisbelow(True)

fig.tight_layout()
save_fig(fig, "slider_probe_counterbalance_exact_within_block", category="memory", also_html=True)
plt.show()

# ============================================================
# Plot 2: pair-level earlier-assignment probability
# ============================================================
# This asks a different question:
#   Across simulated participants, is each distance-2 pair approximately
#   equally likely to be assigned as the earlier vs later probe?

heat = (
    d2_rate_df
    .pivot(index="block", columns="pair_label", values="p_earlier")
    .sort_index()
)

fig, ax = plt.subplots(figsize=(14.5, 4.8))

im = ax.imshow(
    heat.values,
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="RdBu_r"
)

ax.set_xticks(np.arange(heat.shape[1]))
ax.set_xticklabels(heat.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(np.arange(heat.shape[0]))
ax.set_yticklabels([f"Block {b}" for b in heat.index], fontsize=10)

# Annotate cells with p(earlier)
for r in range(heat.shape[0]):
    for c in range(heat.shape[1]):
        val = heat.values[r, c]
        ax.text(
            c,
            r,
            f"{val:.2f}",
            ha="center",
            va="center",
            fontsize=8.5,
            color="white" if val < 0.30 or val > 0.70 else "#222222"
        )

cbar = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.015)
cbar.set_label("Probability of earlier-probe assignment", fontsize=10)

ax.set_title(
    "Pair-level earlier/later assignment across simulated participants",
    fontsize=14,
    fontweight="bold",
    pad=14
)

ax.set_xlabel("Distance-2 memory pair", fontsize=11)
ax.set_ylabel("Block", fontsize=11)

fig.tight_layout()
save_fig(fig, "slider_probe_pair_level_assignment_probability", category="memory", also_html=True)
plt.show()

# ============================================================
# Reviewer-facing HTML summary
# ============================================================
uid = "slider_probe_counterbalance"

summary_html = (
    styled_table_css(uid)
    + f'<div id="{uid}">'
    + "<h2>Slider-Probe Counterbalancing Simulation</h2>"
    + "<p>This simulation mirrors the revised memory-task placement logic: "
      "every memory pair receives a slider probe; one-intervening-item pairs use the only candidate; "
      "two-intervening-item pairs are pseudo-randomized and assigned with exact within-block "
      "earlier/later balance.</p>"
    + f"<p><strong>Simulation:</strong> {N_SIM} participants × 4 blocks.</p>"
    + "<h3>Participant-block invariant summary</h3>"
    + summary_by_block.to_html()
    + "<h3>Distance-2 pair-level assignment rates</h3>"
    + d2_rate_df[[
        "block",
        "pair_id",
        "pair_label",
        "earlier",
        "later",
        "total",
        "p_earlier",
        "p_later",
        "earlier_minus_later",
      ]].to_html(index=False)
    + "</div>"
)

save_html(summary_html, "slider_probe_counterbalance_summary.html", category="memory")

✓ 1000 simulated participants × 4 blocks checked.
✓ Every participant-block has 14 slider trials.
✓ Every participant-block has exact probe balance: 6 center / 4 earlier / 4 later.
✓ Every pair appears once per participant-block.


selected_probe_position center         earlier         later        
                          mean min max    mean min max  mean min max
block                                                               
1                          6.0   6   6     4.0   4   4   4.0   4   4
2                          6.0   6   6     4.0   4   4   4.0   4   4
3                          6.0   6   6     4.0   4   4   4.0   4   4
4                          6.0   6   6     4.0   4   4   4.0   4   4

selected_probe_position,block,pair_id,pair_label,earlier,later,total,p_earlier,p_later,earlier_minus_later
0,1,P02,"P02 (6, 9)",520,480,1000,0.520,0.480,40
1,1,P03,"P03 (7, 10)",514,486,1000,0.514,0.486,28
2,1,P06,"P06 (17, 20)",497,503,1000,0.497,0.503,-6
3,1,P08,"P08 (23, 26)",495,505,1000,0.495,0.505,-10
4,1,P09,"P09 (28, 31)",502,498,1000,0.502,0.498,4
5,1,P10,"P10 (30, 33)",495,505,1000,0.495,0.505,-10
6,1,P12,"P12 (35, 38)",502,498,1000,0.502,0.498,4
7,1,P14,"P14 (42, 45)",475,525,1000,0.475,0.525,-50
8,2,P02,"P02 (6, 9)",504,496,1000,0.504,0.496,8
9,2,P03,"P03 (7, 10)",495,505,1000,0.495,0.505,-10


  ✓ memory_task/slider_probe_counterbalance_exact_within_block.png + .pdf
  ✓ memory_task/slider_probe_counterbalance_exact_within_block.html
  ✓ memory_task/slider_probe_pair_level_assignment_probability.png + .pdf
  ✓ memory_task/slider_probe_pair_level_assignment_probability.html
  ✓ memory_task/slider_probe_counterbalance_summary.html


## H. Pair-order pseudo-randomization simulation

This check validates the randomization of pair order, independent of the slider probe assignment.
> Is memory-test order randomized enough that a given pair is not always tested early or late?

In [6]:
from scipy import stats as sp_stats

N_RAND = 1000
n_pairs = len(PREDEFINED_PAIRS)
EXPECTED_P = 1.0 / n_pairs   # 1/14 ≈ 0.0714
EXPECTED_COUNT = N_RAND * EXPECTED_P

# Simulate pair-order shuffles
pos_matrix = np.zeros((n_pairs, n_pairs))  # pair × position
for pid in range(N_RAND):
    rng = np.random.default_rng(NOTEBOOK_SEED * 1000 + pid)
    order = list(range(n_pairs))
    rng.shuffle(order)
    for pos, pair_idx in enumerate(order):
        pos_matrix[pair_idx, pos] += 1

prop_matrix = pos_matrix / N_RAND  # observed probability
dev_matrix  = prop_matrix - EXPECTED_P  # deviation from expected

pair_labels = [f'[{p[0]},{p[1]}]' for p in PREDEFINED_PAIRS]

# ── Figure: Two panels ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7),
                                gridspec_kw={'width_ratios': [1.1, 1]})

# Left panel: Deviation heatmap (diverging colormap)
vmax = np.abs(dev_matrix).max()
im = ax1.imshow(dev_matrix, aspect='auto', cmap='RdBu_r',
                vmin=-vmax, vmax=vmax, interpolation='nearest')
ax1.set_xticks(range(n_pairs))
ax1.set_xticklabels([f'{i+1}' for i in range(n_pairs)], fontsize=9)
ax1.set_yticks(range(n_pairs))
ax1.set_yticklabels(pair_labels, fontsize=8, fontfamily='monospace')
ax1.set_xlabel('Presentation slot')
ax1.set_ylabel('Pair')
ax1.set_title(f'Deviation from expected P = 1/{n_pairs} ≈ {EXPECTED_P:.4f}\n'
              f'(blue = under-represented, red = over-represented)',
              fontsize=11, fontweight='bold')

# Annotate each cell with the observed proportion
for r in range(n_pairs):
    for c in range(n_pairs):
        val = prop_matrix[r, c]
        ax1.text(c, r, f'{val:.3f}', ha='center', va='center', fontsize=6.5,
                 color='white' if abs(dev_matrix[r, c]) > vmax * 0.55 else '#333')

cb = fig.colorbar(im, ax=ax1, shrink=0.75, pad=0.02)
cb.set_label(f'P(observed) − P(expected)', fontsize=10)

# Right panel: Per-pair uniformity χ² test
chi2_stats = []
for r in range(n_pairs):
    observed = pos_matrix[r, :]
    chi2, p = sp_stats.chisquare(observed, f_exp=[EXPECTED_COUNT] * n_pairs)
    chi2_stats.append({'pair': pair_labels[r], 'χ²': chi2, 'p': p})

chi_df = pd.DataFrame(chi2_stats)
colors = ['#22c55e' if p > 0.05 else '#ef4444' for p in chi_df['p']]

ax2.barh(range(n_pairs), chi_df['χ²'], color=colors, alpha=0.8, edgecolor='white')
ax2.axvline(sp_stats.chi2.ppf(0.95, n_pairs - 1), color='#dc2626', ls='--', lw=1.2,
            label=f'χ² critical (α=.05, df={n_pairs-1})')
ax2.set_yticks(range(n_pairs))
ax2.set_yticklabels(pair_labels, fontsize=8, fontfamily='monospace')
ax2.set_xlabel('χ² statistic')
ax2.set_title(f'Per-pair uniformity test (N={N_RAND})\n'
              f'Green = uniform (p > .05), Red = non-uniform',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=9, loc='lower right')
ax2.invert_yaxis()

plt.tight_layout()
save_fig(fig, "pair_order_pseudorandomization", category="memory", also_html=True)
plt.show()

# Summary
n_pass = (chi_df['p'] > 0.05).sum()
print(f"\n── Uniformity summary ──")
print(f"  Expected P per cell: 1/{n_pairs} = {EXPECTED_P:.4f}")
print(f"  Expected count per cell: {EXPECTED_COUNT:.1f}")
print(f"  Max |deviation|: {np.abs(dev_matrix).max():.4f}")
print(f"  χ² tests: {n_pass}/{n_pairs} pairs pass uniformity (p > .05)")
print(f"  {'✅' if n_pass == n_pairs else '⚠'} "
      f"{'All pairs uniformly distributed across slots.' if n_pass == n_pairs else 'Some pairs show positional bias.'}")


  ✓ memory_task/pair_order_pseudorandomization.png + .pdf
  ✓ memory_task/pair_order_pseudorandomization.html

── Uniformity summary ──
  Expected P per cell: 1/14 = 0.0714
  Expected count per cell: 71.4
  Max |deviation|: 0.0256
  χ² tests: 13/14 pairs pass uniformity (p > .05)
  ⚠ Some pairs show positional bias.


## J. Image assignment to serial positions
Adapts the emoji-ribbon visualization to PNG-based stimuli,
showing image names instead of emoji characters in a color-coded block ribbon.

### Shared thumbnail helpers

In [7]:
from pathlib import Path
import base64
from io import BytesIO
import html
import textwrap
import numpy as np
import pandas as pd

try:
    from PIL import Image, ImageOps
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False
    print("[warning] Pillow is not installed. HTML will fall back to filenames instead of thumbnails.")

# REPO_ROOT comes from js_import (auto-detected), not Path.cwd(),
# so this cell works regardless of the notebook working directory.
STIM_DIR = REPO_ROOT / "static" / "stimuli"

# Retina-style thumbnail settings:
# encode_px = actual embedded PNG size
# display_px = CSS display size
THUMB_ENCODE_PX = 88
THUMB_DISPLAY_PX = 44

RIBBON_ENCODE_PX = 76
RIBBON_DISPLAY_PX = 38

PROBE_ENCODE_PX = 96
PROBE_DISPLAY_PX = 48

_thumb_cache = {}

def stimulus_path_to_abs(path):
    if path is None or pd.isna(path):
        return None
    return REPO_ROOT / str(path)

def stimulus_name_from_path(path):
    if path is None or pd.isna(path):
        return "?"
    return (
        str(path)
        .replace("static/stimuli/", "")
        .replace(".png", "")
    )

def img_to_base64_thumb(path, encode_px=88, canvas_px=None):
    """
    Return a crisp base64 PNG thumbnail data URI.

    encode_px controls actual embedded resolution.
    display size is controlled separately in CSS.

    canvas_px gives all thumbnails a stable square canvas, which keeps
    tables/ribbons aligned even when source PNGs have different aspect ratios.
    """
    if not PIL_AVAILABLE:
        return None

    if path is None or pd.isna(path):
        return None

    if canvas_px is None:
        canvas_px = encode_px

    cache_key = (str(path), int(encode_px), int(canvas_px))
    if cache_key in _thumb_cache:
        return _thumb_cache[cache_key]

    p = stimulus_path_to_abs(path)

    if p is None or not p.exists():
        _thumb_cache[cache_key] = None
        return None

    try:
        im = Image.open(p).convert("RGBA")

        # Fit inside a square canvas without distortion.
        im.thumbnail((encode_px, encode_px), Image.LANCZOS)

        canvas = Image.new("RGBA", (canvas_px, canvas_px), (255, 255, 255, 0))
        x = (canvas_px - im.width) // 2
        y = (canvas_px - im.height) // 2
        canvas.alpha_composite(im, (x, y))

        buf = BytesIO()
        canvas.save(buf, format="PNG", optimize=True)
        encoded = base64.b64encode(buf.getvalue()).decode("utf-8")
        uri = f"data:image/png;base64,{encoded}"

        _thumb_cache[cache_key] = uri
        return uri

    except Exception as e:
        print(f"[thumbnail warning] Could not render {p}: {e}")
        _thumb_cache[cache_key] = None
        return None

def stim_thumb_html(path, name=None, encode_px=88, display_px=44, show_name=True, css_class="stim-thumb"):
    """
    HTML thumbnail. Uses a 2x-ish embedded PNG for sharper CSS display.
    """
    if name is None:
        name = stimulus_name_from_path(path)

    safe_name = html.escape(str(name))
    uri = img_to_base64_thumb(path, encode_px=encode_px, canvas_px=encode_px)

    if uri is None:
        return f'<div class="stim-fallback">{safe_name}</div>'

    label_html = f'<div class="stim-name">{safe_name}</div>' if show_name else ""

    return (
        f'<div class="stim-thumb-wrap" style="min-width:{display_px + 14}px;">'
        f'<img class="{css_class}" src="{uri}" alt="{safe_name}" title="{safe_name}" '
        f'style="width:{display_px}px;height:{display_px}px;"/>'
        f'{label_html}'
        f'</div>'
    )

def get_row_value(row, *names, default=None):
    for name in names:
        if name in row.index:
            return row[name]
    return default

def pair_from_row_value(x):
    if isinstance(x, (tuple, list)):
        return tuple(int(v) for v in x)

    s = str(x).strip()
    s = s.replace("[", "").replace("]", "").replace("(", "").replace(")", "")
    parts = [p.strip() for p in s.split(",") if p.strip()]
    return tuple(int(p) for p in parts)

def gimg_from_block(blk_imgs, trial_num):
    row = blk_imgs[blk_imgs.trial == int(trial_num)]
    if len(row) == 0:
        return {"path": None, "name": "?"}

    return {
        "path": row.iloc[0]["path"],
        "name": row.iloc[0]["name"],
    }

thumb_css = textwrap.dedent(f"""\
<style>
.stim-table-doc {{
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Arial, sans-serif;
  color: #222;
  background: #fff;
  padding: 18px 12px;
}}
.stim-table-doc h1,
.stim-table-doc h2,
.stim-table-doc h3 {{
  margin: 0 0 12px 0;
}}
.stim-table-doc p {{
  max-width: 980px;
  font-size: 13px;
  line-height: 1.45;
  color: #475569;
}}
.stim-table-doc table {{
  border-collapse: collapse;
  font-size: 12px;
  width: 100%;
  max-width: 1500px;
}}
.stim-table-doc caption {{
  caption-side: top;
  text-align: left;
  font-weight: 750;
  padding: 0 0 8px 0;
}}
.stim-table-doc th {{
  background: #f4f6f8;
  border: 1px solid #d0d7de;
  padding: 6px 8px;
  text-align: left;
  font-weight: 750;
}}
.stim-table-doc td {{
  border: 1px solid #d0d7de;
  padding: 5px 7px;
  vertical-align: middle;
}}
.stim-thumb-wrap {{
  display: flex;
  flex-direction: column;
  align-items: center;
  gap: 2px;
}}
.stim-thumb {{
  object-fit: contain;
  border-radius: 8px;
  background: #f8fafc;
  border: 1px solid #d8dee9;
  padding: 3px;
  image-rendering: auto;
}}
.stim-name {{
  font-size: 9px;
  max-width: 88px;
  line-height: 1.05;
  text-align: center;
  color: #334155;
  word-break: break-word;
}}
.stim-fallback {{
  font-size: 10px;
  max-width: 90px;
  word-break: break-word;
  color: #475569;
}}
</style>
""")

# -----------------------------
# Simulate one participant-level image assignment
# -----------------------------
assert len(STIMULUS_IMAGE_POOL) >= 200, "STIMULUS_IMAGE_POOL must contain at least 200 paths."

rng_j = np.random.default_rng(NOTEBOOK_SEED)
shuffled_pool = STIMULUS_IMAGE_POOL.copy()
rng_j.shuffle(shuffled_pool)

img_rows = []

for blk in range(4):
    for t in range(50):
        gi = blk * 50 + t
        path = shuffled_pool[gi]
        abs_path = stimulus_path_to_abs(path)

        img_rows.append(dict(
            global_trial=gi + 1,
            block=blk + 1,
            trial=t + 1,
            path=path,
            name=stimulus_name_from_path(path),
            file_exists=abs_path.exists() if abs_path else False,
        ))

img_df = pd.DataFrame(img_rows)

display(img_df.head(20))

assert len(img_df) == 200
assert img_df["path"].nunique() == 200
assert img_df["file_exists"].all(), "Some stimulus PNG files are missing under static/stimuli/."

print("✓ Shared image-assignment setup complete.")
print("✓ 200 unique PNG paths assigned to 4 blocks × 50 trials.")

,global_trial,block,trial,path,name,file_exists
0,1,1,1,static/stimuli/shield.png,shield,True
1,2,1,2,static/stimuli/bowling.png,bowling,True
2,3,1,3,static/stimuli/newspaper.png,newspaper,True
3,4,1,4,static/stimuli/clutch.png,clutch,True
4,5,1,5,static/stimuli/carp-streamer.png,carp-streamer,True
5,6,1,6,static/stimuli/auto-rickshaw.png,auto-rickshaw,True
6,7,1,7,static/stimuli/goggles.png,goggles,True
7,8,1,8,static/stimuli/sunflower.png,sunflower,True
8,9,1,9,static/stimuli/saxophone.png,saxophone,True
9,10,1,10,static/stimuli/crown.png,crown,True


✓ Shared image-assignment setup complete.
✓ 200 unique PNG paths assigned to 4 blocks × 50 trials.


### Stimulus ribbons html

Example participant stimulus ribbon under a fixed notebook seed.

What is universal across participants:
- block structure
- memory-pair endpoint positions
- candidate probe positions
- change-point positions

What varies across participants:
- which PNG object appears at trial 1, trial 2, ..., trial 200

In [8]:
# ============================================================
# Export 1: stimulus_ribbons.html
# ============================================================

ribbon_css = textwrap.dedent(f"""\
<style>
.ribbon-doc {{
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Arial, sans-serif;
  color: #222;
  background: #fff;
  padding: 18px 8px;
}}
.ribbon-doc h1 {{
  font-size: 18px;
  font-weight: 750;
  margin: 0 0 8px 0;
}}
.ribbon-doc .note {{
  max-width: 980px;
  font-size: 12px;
  line-height: 1.45;
  color: #475569;
  margin: 0 0 12px 0;
}}
.ribbon-doc .legend {{
  display: flex;
  flex-wrap: wrap;
  gap: 6px 14px;
  align-items: center;
  font-size: 11.5px;
  color: #333;
  margin: 8px 0 14px 0;
}}
.ribbon-doc .legend-item {{
  display: inline-flex;
  align-items: center;
  gap: 5px;
  white-space: nowrap;
}}
.ribbon-doc .swatch {{
  display: inline-block;
  width: 16px;
  height: 12px;
  border-radius: 2px;
}}
.ribbon-doc .swatch.tested {{
  background: #fff3a3;
  border: 1px solid #b58a00;
}}
.ribbon-doc .swatch.probe {{
  background: #cfe8ff;
  border: 1px solid #2b6cb0;
}}
.ribbon-doc .swatch.cp-line {{
  width: 3px;
  height: 16px;
  background: #d62728;
}}
.ribbon-doc .ribbon-title {{
  font-weight: 700;
  font-size: 12.5px;
  margin: 14px 0 4px 0;
}}
.ribbon-doc table.ribbon {{
  border-collapse: collapse;
  table-layout: fixed;
  width: 100%;
  max-width: 1650px;
}}
.ribbon-doc table.ribbon td {{
  width: 2%;
  height: 68px;
  border: 1px solid #d2d2d2;
  text-align: center;
  vertical-align: middle;
  position: relative;
  background: #fff;
  overflow: hidden;
  padding: 2px;
}}
.ribbon-doc table.ribbon td.tested {{
  background: #fff3a3;
  border-color: #b58a00;
}}
.ribbon-doc table.ribbon td.probe {{
  background: #cfe8ff;
  border-color: #2b6cb0;
}}
.ribbon-doc table.ribbon td.cp-after {{
  border-right: 3px solid #d62728;
}}
.ribbon-doc .pair-label {{
  position: absolute;
  top: 1px;
  left: 1px;
  font-size: 7px;
  font-weight: 800;
  color: #5b4400;
  background: rgba(255,255,255,.88);
  border-radius: 2px;
  padding: 0 2px;
}}
.ribbon-doc .probe-label {{
  position: absolute;
  top: 1px;
  right: 1px;
  font-size: 7px;
  font-weight: 800;
  color: #004b88;
  background: rgba(255,255,255,.88);
  border-radius: 2px;
  padding: 0 2px;
}}
.ribbon-doc .trial-num {{
  position: absolute;
  bottom: 1px;
  right: 2px;
  font-size: 7px;
  color: #777;
}}
.ribbon-doc .ribbon-img {{
  width: {RIBBON_DISPLAY_PX}px;
  height: {RIBBON_DISPLAY_PX}px;
  object-fit: contain;
  display: block;
  margin: 15px auto 0 auto;
  image-rendering: auto;
}}
.ribbon-doc .missing {{
  font-size: 8px;
  color: #64748b;
  margin-top: 24px;
}}
</style>
""")

# Build pair endpoint lookup and candidate/probe lookup.
pair_endpoints = set()
pair_labels = {}
candidate_positions = set()

for pi, pair in enumerate(PREDEFINED_PAIRS):
    ep1, ep2 = pair

    for ep in [ep1, ep2]:
        pair_endpoints.add(ep)
        pair_labels.setdefault(ep, []).append(f"P{pi+1:02d}")

    for c in range(ep1 + 1, ep2):
        candidate_positions.add(c)

ribbon_body = [
    '<div class="ribbon-doc">',
    '<h1>DroneTask stimulus</h1>',
    '<p class="note">Each cell is one encoding trial. Yellow marks memory-pair endpoints; blue marks slider-probe positions; red borders mark change-point boundaries. </p>',
    '<div class="legend">',
    '<span class="legend-item"><span class="swatch tested"></span>memory-pair endpoint item</span>',
    '<span class="legend-item"><span class="swatch probe"></span>slider-probe item</span>',
    '<span class="legend-item"><span class="swatch cp-line"></span>change-point boundary</span>',
    '</div>'
]

for blk in range(1, 5):
    vol = int(factors_vol[blk - 1])
    cps = set(CP_HIGH_VOL if vol == 49 else CP_LOW_VOL)
    blk_imgs = img_df[img_df.block == blk].sort_values("trial")

    ribbon_body.append(
        f'<div class="ribbon-title">Block {blk} '
        f'({html.escape(str(factors_valence[blk - 1]))}, '
        f'{"high" if vol == 49 else "low"} volatility)</div>'
    )

    ribbon_body.append('<table class="ribbon"><tr>')

    for t in range(1, 51):
        classes = []
        labels = ""

        if t in pair_endpoints:
            classes.append("tested")

        if t in candidate_positions:
            classes.append("probe")

        if t in cps:
            classes.append("cp-after")

        if t in pair_labels:
            labels += f'<span class="pair-label">{html.escape(",".join(pair_labels[t]))}</span>'

        if t in candidate_positions:
            labels += '<span class="probe-label">PRB</span>'

        row_t = blk_imgs[blk_imgs.trial == t]

        if len(row_t) > 0:
            path = row_t.iloc[0]["path"]
            name = row_t.iloc[0]["name"]
            uri = img_to_base64_thumb(path, encode_px=RIBBON_ENCODE_PX, canvas_px=RIBBON_ENCODE_PX)
        else:
            name, uri = "?", None

        safe_name = html.escape(str(name))
        cls = f' class="{" ".join(classes)}"' if classes else ""

        if uri:
            stim_content = (
                f'<img class="ribbon-img" src="{uri}" alt="{safe_name}" title="{safe_name}"/>'
            )
        else:
            stim_content = f'<div class="missing">{safe_name}</div>'

        ribbon_body.append(
            f'<td{cls}>{labels}{stim_content}<span class="trial-num">{t}</span></td>'
        )

    ribbon_body.append("</tr></table>")

ribbon_body.append("</div>")

save_html(
    ribbon_css + "\n".join(ribbon_body),
    "stimulus_ribbons.html"
, category="memory")

print("✓ Exported stimulus_ribbons.html.")

  ✓ memory_task/stimulus_ribbons.html
✓ Exported stimulus_ribbons.html.


### Items to serial position example

Item assignment to serial positions for one simulated participant. The actual task shuffles STIMULUS_IMAGE_POOL per participant.

In [9]:
# ============================================================
# Export 2: items_to_serial_positions.html
# ============================================================

imgassign_rows = []

for _, r in img_df.head(50).iterrows():
    imgassign_rows.append(
        f"<tr>"
        f"<td>{int(r.global_trial)}</td>"
        f"<td>{int(r.block)}</td>"
        f"<td>{int(r.trial)}</td>"
        f"<td>{stim_thumb_html(r.path, r['name'], encode_px=THUMB_ENCODE_PX, display_px=THUMB_DISPLAY_PX, show_name=False)}</td>"
        f"<td>{html.escape(str(r['name']))}</td>"
        f"<td>{html.escape(str(r.path))}</td>"
        f"</tr>"
    )

save_html(
    thumb_css
    + styled_table_css("imgassign")
    + '<div id="imgassign" class="stim-table-doc">'
    + "<h2>Items assignment to serial positions</h2>"
    + "<table>"
    + "<caption>Items to serial positions: first block / first 50 rows</caption>"
    + "<tr><th>global_trial</th><th>block</th><th>within_block_trial</th><th>thumbnail</th><th>name</th><th>path</th></tr>"
    + "".join(imgassign_rows)
    + "</table></div>",
    "items_to_serial_positions.html"
, category="memory")

display(img_df.head(50)[["global_trial", "block", "trial", "name", "path", "file_exists"]])

print("✓ Exported items_to_serial_positions.html.")

  ✓ memory_task/items_to_serial_positions.html


,global_trial,block,trial,name,path,file_exists
0,1,1,1,shield,static/stimuli/shield.png,True
1,2,1,2,bowling,static/stimuli/bowling.png,True
2,3,1,3,newspaper,static/stimuli/newspaper.png,True
3,4,1,4,clutch,static/stimuli/clutch.png,True
4,5,1,5,carp-streamer,static/stimuli/carp-streamer.png,True
5,6,1,6,auto-rickshaw,static/stimuli/auto-rickshaw.png,True
6,7,1,7,goggles,static/stimuli/goggles.png,True
7,8,1,8,sunflower,static/stimuli/sunflower.png,True
8,9,1,9,saxophone,static/stimuli/saxophone.png,True
9,10,1,10,crown,static/stimuli/crown.png,True


✓ Exported items_to_serial_positions.html.


### memory_probe_assignment.html

The pair structure is universal, but the specific endpoint/probe items and the earlier/later probe item for the distance-2 pairs vary across participants.

In [10]:
# ============================================================
# Export 3: memory_probe_image_assignment.html
# ============================================================

required_sim_cols_any = (
    ("selected_probe_index" in sim_df.columns) or ("probe_idx" in sim_df.columns)
)
assert required_sim_cols_any, "sim_df must include selected_probe_index or probe_idx from the counterbalancing simulation cell."

probe_img_rows = []

for blk in range(1, 5):
    blk_imgs = img_df[img_df.block == blk].sort_values("trial")

    p0_blk = (
        sim_df[(sim_df.sim_participant == 0) & (sim_df.block == blk)]
        .copy()
        .sort_values("pair_id")
    )

    for _, r in p0_blk.iterrows():
        pair = pair_from_row_value(r["pair"])
        ep1, ep2 = pair

        probe_idx = get_row_value(r, "selected_probe_index", "probe_idx")
        probe_label = get_row_value(r, "selected_probe_position", "probe_label", default="?")

        first = gimg_from_block(blk_imgs, ep1)
        probe = gimg_from_block(blk_imgs, probe_idx)
        second = gimg_from_block(blk_imgs, ep2)

        probe_img_rows.append(dict(
            block=blk,
            pair_id=r.pair_id,
            pair=str(pair),
            ep1_idx=ep1,
            ep1_img=first["name"],
            ep1_path=first["path"],
            probe_idx=int(probe_idx),
            probe_img=probe["name"],
            probe_path=probe["path"],
            ep2_idx=ep2,
            ep2_img=second["name"],
            ep2_path=second["path"],
            probe_label=probe_label,
        ))

probe_img_df = pd.DataFrame(probe_img_rows)

probe_display_cols = [
    "block",
    "pair_id",
    "pair",
    "ep1_idx",
    "ep1_img",
    "probe_idx",
    "probe_img",
    "ep2_idx",
    "ep2_img",
    "probe_label",
]

display(probe_img_df[probe_display_cols])

probe_img_html_rows = []

for _, row in probe_img_df.iterrows():
    probe_img_html_rows.append(
        f"<tr>"
        f"<td>{row['block']}</td>"
        f"<td>{html.escape(str(row['pair_id']))}</td>"
        f"<td>{html.escape(str(row['pair']))}</td>"
        f"<td>{row['ep1_idx']}</td>"
        f"<td>{stim_thumb_html(row['ep1_path'], row['ep1_img'], encode_px=PROBE_ENCODE_PX, display_px=PROBE_DISPLAY_PX, show_name=True)}</td>"
        f"<td>{row['probe_idx']}</td>"
        f"<td>{stim_thumb_html(row['probe_path'], row['probe_img'], encode_px=PROBE_ENCODE_PX, display_px=PROBE_DISPLAY_PX, show_name=True)}</td>"
        f"<td>{row['ep2_idx']}</td>"
        f"<td>{stim_thumb_html(row['ep2_path'], row['ep2_img'], encode_px=PROBE_ENCODE_PX, display_px=PROBE_DISPLAY_PX, show_name=True)}</td>"
        f"<td>{html.escape(str(row['probe_label']))}</td>"
        f"</tr>"
    )

save_html(
    thumb_css
    + styled_table_css("probeimg")
    + '<div id="probeimg" class="stim-table-doc">'
    + "<h2>Memory probe assignment</h2>"
    + "<p>Each row shows the two endpoint items and the selected slider-probe item for one simulated participant. </p>"
    + "<table>"
    + "<caption>Memory probe assignment</caption>"
    + "<tr>"
    + "<th>block</th><th>pair_id</th><th>pair</th>"
    + "<th>endpoint 1 index</th><th>endpoint 1 item</th>"
    + "<th>probe index</th><th>selected probe item</th>"
    + "<th>endpoint 2 index</th><th>endpoint 2 item</th>"
    + "<th>probe role</th>"
    + "</tr>"
    + "".join(probe_img_html_rows)
    + "</table></div>",
    "memory_probe_assignment.html"
, category="memory")

print("✓ Exported memory_probe_assignment.html.")

,block,pair_id,pair,ep1_idx,ep1_img,probe_idx,probe_img,ep2_idx,ep2_img,probe_label
0,1,P01,"(2, 4)",2,bowling,3,newspaper,4,clutch,center
1,1,P02,"(6, 9)",6,auto-rickshaw,7,goggles,9,saxophone,earlier
2,1,P03,"(7, 10)",7,goggles,9,saxophone,10,crown,later
3,1,P04,"(11, 13)",11,black-nib,12,telescope,13,microscope,center
4,1,P05,"(16, 18)",16,yarn,17,watch,18,card-file-box,center
5,1,P06,"(17, 20)",17,watch,19,robot,20,computer-disk,later
6,1,P07,"(22, 24)",22,violin,23,sled,24,popcorn,center
7,1,P08,"(23, 26)",23,sled,25,video-camera,26,shopping-bags,later
8,1,P09,"(28, 31)",28,wrapped-gift,29,satellite,31,funeral-urn,earlier
9,1,P10,"(30, 33)",30,spiral-shell,32,fire,33,mate,later


  ✓ memory_task/memory_probe_assignment.html
✓ Exported memory_probe_assignment.html.
